# Changes from ConstantThicknessProjection.ipynb

 - Gets rid of the gap between bone and cartilage that gets stitched together in previous notebook.
 - Updates the method and postprocessing to match ArticularGap method.
 - Cartilage region dictated by distance from articulating bone instead of segmented boundary loop

In [1]:
import numpy as np
import pandas as pd
import pyvista as pv
import gdist
from scipy.spatial.distance import cdist
from pathlib import Path
import subprocess

from phd_helpers.CartilageGeneration import (
    mesh_checks, bone_cartilage_checks, get_outward_normal_mask, flip_faces, taper_f, remove_normals, get_min_df_fast
)
from phd_helpers.paths import find_corresponding_cells, identical_points_count, get_subject_stl_path, get_boundary


In [2]:
# INPUTS #

subject, sideL = '14548', 'R'
bone, ar_bone = 'tpm', 'mc1'
mesh_id = '-0'

stl_path = get_subject_stl_path(subject, sideL)

output_root = 'testing3'
output_dir = Path(f"../MeshPrep3/{output_root}/meshes/{subject}{sideL}/{bone}-{ar_bone}")
output_path = output_dir / '2Dmesh'

quality_path = ''
compute_quality = False

bone_mesh = pv.read(output_path / f'remesh{mesh_id}.obj')
ar_path = output_dir.parent / f"{ar_bone}-{bone}/2Dmesh/remesh{mesh_id}.obj"
arbone_mesh = pv.read(ar_path)

# compute min df
max_gap_cartilage = 2
poses = ['adduction','abduction','flexion','extension','pinch','grasp','jar','neutral']
min_df = get_min_df_fast(stl_path, bone, ar_bone, bone_mesh, arbone_mesh, poses, max_gap_cartilage)

100%|██████████| 8/8 [00:02<00:00,  3.15it/s]


In [3]:
remesh_cartilage = True
cart_remesh_name = f'CartilageCap.obj' # cartilage cap mesh file name
cart_remesh_input_path = f'/Users/matty/Documents/CPPremesh/CGALremesh/FB-inputMesh/{cart_remesh_name}' # path to c++ fixed boundary input

taper_width = 1 # width of cartilage taper region
target_height = 1 # target cartilage height in central region
p_h = 2 # shape of taper height (1 = linear , higher = steeper taper)

smooth_iters = 100 # need to look at this, currently uses laplacian 
edge_length = 0.2 # target edge length of cartilage remesh
n_iters = 5 # n isotropic remeshing iterations for cartilage remesh



In [4]:
bone_mesh['point_id'] = np.arange(bone_mesh.n_points)
bone_mesh['cell_id'] = np.arange(bone_mesh.n_cells)

# extract cartilage region from bone mesh
cart_surf = bone_mesh.extract_points(min_df['bone_id'], adjacent_cells=False).extract_geometry()
cart_surf['region_id'] = np.full(cart_surf.n_cells, 2) # label cartilage region
cart_surf['cart_point_id'] = np.arange(cart_surf.n_points)
cart_surf['cart_cell_id'] = np.arange(cart_surf.n_cells)

bone_mesh['region_id'] = np.ones(bone_mesh.n_cells, dtype=int)
bone_mesh['region_id'][cart_surf['cell_id']] = 3 # label interface region on bone mesh

# measure distances of cartilage points from the boundary to determine taper region
cart_boundary = get_boundary(cart_surf)
boundary_dists = gdist.compute_gdist(
    cart_surf.points.astype(np.float64),
    cart_surf.faces.reshape(-1, 4)[:, 1:].astype(np.int32),
    source_indices=cart_boundary['cart_point_id'].astype(np.int32) 
)

taper_mask = boundary_dists <= taper_width # mask on cart_surf points
taper_heights = taper_f(boundary_dists, taper_width, target_height, p=p_h)

# extrude points computed distances in normal directions
cart_normals = cart_surf.compute_normals(point_normals=True).point_data['Normals']
cart_surf.points[~taper_mask] += cart_normals[~taper_mask] * target_height
cart_surf.points[taper_mask] += cart_normals[taper_mask] * taper_heights[taper_mask].reshape(-1, 1)

# assign array for inner(1) and taper(0) cartilage cells - taper region is only cells with no taper points
cart_surf['inner_cells'] = np.zeros(cart_surf.n_cells, dtype=int)
inner_region = cart_surf.extract_points(~taper_mask, adjacent_cells=False)
cart_surf['inner_cells'][inner_region['cart_cell_id']] = 1
bone_mesh['inner_cells'] = np.full(bone_mesh.n_cells, -1)



# check for flat faces #
_, ps = bone_mesh.find_closest_cell(cart_surf.points, return_closest_point=True)
implicit_distances = np.linalg.norm(cart_surf.points - ps, axis=1) #*** changed from line above

mesh_faces = cart_surf.faces.reshape(-1, 4)[:, 1:]
flat_face_mask = (implicit_distances[mesh_faces] <= 1e-12).all(axis=1) #*** changed from ==0 to <=1e-12

# remove flat faces and leftover points - occur when all 3 vertices lie on the boundary
mesh_clean = pv.PolyData(cart_surf.points, cart_surf.faces.reshape(-1, 4)[~flat_face_mask]).compute_normals(auto_orient_normals=True)
mesh_clean.remove_unused_points(inplace=True)
mesh_clean['mesh_clean_id'] = np.arange(mesh_clean.n_points)
# get array of inner points/cells on mesh_clean 
mesh_clean['inner_cells'] = cart_surf['inner_cells'][find_corresponding_cells(cart_surf, mesh_clean)]


# get edge points on mesh and bone_mesh
mesh_clean_edge = get_boundary(mesh_clean)
mesh_clean_edge_mask = np.isin(mesh_clean['mesh_clean_id'], mesh_clean_edge['mesh_clean_id']) # on mesh_clean
mesh_edge_ids = mesh_clean['mesh_clean_id'][mesh_clean_edge_mask] # on mesh_clean
mesh_edge_points = mesh_clean.points[mesh_clean_edge_mask] # on mesh_clean

_, bone_mesh_edge_ids, _ = identical_points_count(bone_mesh.points, mesh_edge_points, return_indices=True)
bone_mesh_edge_ids = np.sort(bone_mesh_edge_ids)

if len(bone_mesh_edge_ids) != len(mesh_edge_points):
    raise AssertionError("Mesh boundary has changed")

if (cart_surf.n_cells - mesh_clean.n_cells) != flat_face_mask.sum():
    raise AssertionError("Not all flat faces removed")



# smooth cartilage surface # - fixed boundary 
cart_smooth = mesh_clean.smooth(
n_iter=smooth_iters,
feature_angle=180, # prevent anything from being feature cos I don't understant feature_smoothing arg...
boundary_smoothing = False, # keeps boundary fixed-ish
feature_smoothing = False # prevents feature edges from being identified - so they get smoothed?... (idk stupid)
)
cart_smooth_edge = get_boundary(cart_smooth)

if cart_smooth_edge.n_points != mesh_clean_edge.n_points:
    raise AssertionError(f'{cart_smooth_edge.n_points - mesh_clean_edge.n_points} Boundary points lost during smoothing')

# put edge points back so they are numerically identical
cart_smooth.points[mesh_edge_ids] = bone_mesh.points[bone_mesh_edge_ids]

if compute_quality:
    # measure change due to smoothing and save to file - compute_implicit_distance is fastest algo
    cart_smooth['implicit_distance_orig'] = np.asarray(
        cart_smooth.compute_implicit_distance(mesh_clean)['implicit_distance'],
        dtype=np.float64
        ).copy()
    quality_path.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        'dist_orig':cart_smooth['implicit_distance_orig'], 
        'inner_point':cart_smooth['inner_points']
        }).to_csv(quality_path / 'smoothing_dists.csv', index=False)


if remesh_cartilage:
    ################# REMESH CARTILAGE #################
    # remesh cartilage with CGAL fixed boundary - check boundary hasn't moved (and put points back anyway)
    # this is the final step in creating the cartilage cap mesh (just need to attach to bone after this - trivial)

    print('Writing mesh to remeshing input directory')
    cart_smooth.save(cart_remesh_input_path)

    print('Remeshing cartilage cap')

    build_dir = Path("/Users/matty/Documents/CPPremesh/CGALremesh/build")
    exe = build_dir / "fixed_boundary"

    cart_remesh_output_path = cart_remesh_input_path.replace('-input', '-output')
    args = [
        str(exe),
        str(cart_remesh_input_path),  # path to input mesh
        str(cart_remesh_output_path), # path to output mesh
        str(edge_length), # target edge length
        str(n_iters), # number of CGAL isotropic remeshing iterations
        ]

    result = subprocess.run(args, cwd=str(build_dir), text=True)
    result.check_returncode()  # raise after printing




    # load remeshed cartilage cap
    cart_remesh = pv.read(cart_remesh_output_path)
    cart_remesh['remesh_ids'] = np.arange(cart_remesh.n_points)

    # check boudnary hasn't moved
    cart_remesh_edge = get_boundary(cart_remesh)
    if cart_remesh_edge.n_points != cart_smooth_edge.n_points:
        raise ValueError(f'{cart_remesh_edge.n_points - cart_smooth_edge.n_points} Boundary points lost in remeshing')
    
    remesh_edge_dists = cdist(cart_smooth_edge.points, cart_remesh_edge.points)
    cart_smooth_edge_ids = np.argmin(remesh_edge_dists, axis=0)
    closest_dists = np.min(remesh_edge_dists, axis=0)
    if (closest_dists > 1e-5).any():
        raise AssertionError(f"Cartilage boundary points moved during remeshing: {max(closest_dists):.5f} mm")

    # put edge points back so they are numerically identical
    cart_remesh.points[cart_remesh_edge['remesh_ids']] = cart_smooth_edge.points[cart_smooth_edge_ids]

    # measure cartilage height
    cart_remesh_height = cart_remesh.compute_implicit_distance(bone_mesh)['implicit_distance']
    if (cart_remesh_height < 0).any():
        raise AssertionError('Not all cartilage points above bone surface')

    # add cartilge taper/inner region array
    cart_remesh['inner_cells'] = cart_smooth['inner_cells'][cart_smooth.find_closest_cell(cart_remesh.cell_centers().points)]
    
    if compute_quality:
        # measure change due to remeshing and save to file
        cart_remesh['implicit_distance_orig'] = np.asarray(
            cart_remesh.compute_implicit_distance(mesh_clean)['implicit_distance'],
            dtype=np.float64
            ).copy()
        cart_remesh['implicit_distance_smooth'] = np.asarray(
            cart_remesh.compute_implicit_distance(cart_smooth)['implicit_distance'],
            dtype=np.float64
            ).copy()
        pd.DataFrame({
            'dist_orig':cart_remesh['implicit_distance_orig'], 
            'dist_smooth':cart_remesh['implicit_distance_smooth'],
            }).to_csv(quality_path / 'remesh_dists.csv', index=False)
    ################# REMESH CARTILAGE #################
else:
    cart_remesh = cart_smooth.copy(deep=True)
    cart_remesh['remesh_ids'] = np.arange(cart_remesh.n_points)
    cart_remesh_edge = get_boundary(cart_remesh)

print('\nAttaching cartilage surf')
# find which inter_boundary points are still in mesh boundary - after unused point removal
interface_surf = bone_mesh.extract_cells(np.where(bone_mesh['region_id']==3)[0])
interface_boundary = get_boundary(interface_surf)
_, inter_boundary_ids, _ = identical_points_count(interface_boundary.points, mesh_edge_points, return_indices=True)

# find which interface points are still in mesh and their bone_ids
unused_inter_ids = interface_boundary['point_id'][~np.isin(np.arange(interface_boundary.n_points), inter_boundary_ids)]
interface_bone_mesh_ids = interface_surf['point_id'][~np.isin(interface_surf['point_id'], unused_inter_ids)]

# extract bone_cartilage interface mesh
interface_mesh = bone_mesh.extract_points(interface_bone_mesh_ids, adjacent_cells=False).extract_geometry()
################# GET FINAL INTERFACE MESH BETWEEN BONE AND CARTILAGE #################


################# GET FINAL COMBINED MESH WITH REGION IDs #################

# re-assign scalar data - region_id
bone_mesh['region_id'] = np.ones(bone_mesh.n_cells, dtype=int)
bone_mesh['region_id'][find_corresponding_cells(bone_mesh, interface_mesh, raise_error=True)] = 3
cart_remesh['region_id'] = np.full(cart_remesh.n_cells, 2)

# create combined mesh
bone_mesh['inner_cells'] = np.full(bone_mesh.n_cells, -1)
combined_mesh = bone_mesh + cart_remesh
combined_mesh.compute_normals(inplace=True, consistent_normals=False)

combined_mesh['inner_points'] = np.full(combined_mesh.n_points, -1)
combined_mesh['inner_points'][np.unique(combined_mesh.faces.reshape(-1, 4)[:, 1:][np.where(combined_mesh['inner_cells']==1)[0]])] = 1
combined_mesh['inner_points'][np.unique(combined_mesh.faces.reshape(-1, 4)[:, 1:][np.where(combined_mesh['inner_cells']==0)[0]])] = 0

dupe_check = combined_mesh.n_points == (cart_remesh.n_points+bone_mesh.n_points) - mesh_edge_ids.shape[0]
if not dupe_check:
    raise AssertionError('Not all duplicate points removed')
################# GET FINAL COMBINED MESH WITH REGION IDs #################

Writing mesh to remeshing input directory
Remeshing cartilage cap
Remeshing complete.

Attaching cartilage surf


In [8]:
combined_mesh.plot(scalars='inner_cells')

Widget(value='<iframe src="http://localhost:52577/index.html?ui=P_0x17e6c7320_2&reconnect=auto" class="pyvista…

In [6]:
################# MESH CHECKS #################
# get cells ids of each region on the combined mesh
combined_mesh_cell_ids = np.arange(combined_mesh.n_cells)
bone_surf_ids = combined_mesh_cell_ids[np.where(combined_mesh['region_id']==1)[0]]
bone_shell_ids = combined_mesh_cell_ids[np.where(combined_mesh['region_id']!=2)[0]]
cartilage_surf_ids = combined_mesh_cell_ids[np.where(combined_mesh['region_id']==2)[0]]
#interface_surf_ids = combined_mesh_cell_ids[np.where(combined_mesh['region_id']==3)[0]]

# extract enclosed cartilage volume to do checks
cartilage_mesh = combined_mesh.extract_cells(bone_surf_ids, invert=True).extract_geometry()
remove_normals(cartilage_mesh)  # have to remove them before recomputing 
                        #- cos inherited from parent mesh and doesn't recompute something about them for some reason
cartilage_mesh.compute_normals(auto_orient_normals=True, inplace=True)
# check if normals point outwards or inwards and if not flip them
if not get_outward_normal_mask(cartilage_mesh.cell_centers().points, cartilage_mesh.cell_normals, cartilage_mesh).any():
    cartilage_mesh = flip_faces(cartilage_mesh, np.arange(cartilage_mesh.n_cells))
    print('flipped faces')

print('\nMESH CHECKS...')
# mesh checks
print('\n----- CARTILAGE -----')
mesh_checks(cartilage_mesh, raise_error=True)

print('\n----- BONE -----')
mesh_checks(bone_mesh, raise_error=True)

print('\n----- BONE-CARTILAGE -----')
interface_surf = bone_mesh.extract_cells(np.where(bone_mesh['region_id']==3)[0])
bone_cartilage_checks(bone_mesh, cartilage_mesh, interface_surf.points, raise_error=True, check_intersection=False)

combined_mesh_centres = combined_mesh.cell_centers().points
combined_mesh_normals = combined_mesh.cell_normals

# check if normals point outwards
bone_normal_mask = get_outward_normal_mask( # including boundary
    combined_mesh_centres[bone_shell_ids], 
    combined_mesh_normals[bone_shell_ids], 
    bone_mesh
    )
cartilage_normal_mask = get_outward_normal_mask(
    combined_mesh_centres[cartilage_surf_ids], 
    combined_mesh_normals[cartilage_surf_ids], 
    cartilage_mesh
    )

# check if all normals point outwards, fix if not
flip_ids = np.hstack((cartilage_surf_ids[~cartilage_normal_mask], bone_shell_ids[~bone_normal_mask]))
if len(flip_ids):
    combined_mesh_flipped = flip_faces(combined_mesh, flip_ids)
    # changed flip faces to carry over point/cell arrays 2026-03-11
else:
    combined_mesh_flipped = combined_mesh

# check normal direction
if not get_outward_normal_mask( 
    combined_mesh_centres[bone_shell_ids], 
    combined_mesh_flipped.cell_normals[bone_shell_ids], 
    bone_mesh
    ).all():
    raise AssertionError('Not all bone normals point outward')
if not get_outward_normal_mask( 
    combined_mesh_centres[cartilage_surf_ids], 
    combined_mesh_flipped.cell_normals[cartilage_surf_ids], 
    cartilage_mesh
    ).all():
    raise AssertionError('Not all cartilage surface normals point outward')

# manually check normals
faces = combined_mesh_flipped.faces.reshape(-1, 4)[:, 1:]
points = combined_mesh_flipped.points

v1 = points[faces[:, 1]] - points[faces[:, 0]]
v2 = points[faces[:, 2]] - points[faces[:, 0]]
geom_normals = np.cross(v1, v2)
geom_normals /= np.linalg.norm(geom_normals, axis=1)[:, None]

dots = np.einsum("ij,ij->i", geom_normals, combined_mesh_flipped['Normals'])
if not np.min(dots)>0.999:
    raise AssertionError("['Normals'] don't match edge winding")
# mesh['Normals']doesn't update with edge winding but .cell_normals does
################# MESH CHECKS #################


MESH CHECKS...

----- CARTILAGE -----
Trimesh checks:
Mesh is watertight               True
Mesh is winding consistent       True

PyVista checks:
Mesh is manifold (no open edges) True
All normals point outwards:      True
No duplicate faces               True
No duplicate points              True

----- BONE -----
Trimesh checks:
Mesh is watertight               True
Mesh is winding consistent       True

PyVista checks:
Mesh is manifold (no open edges) True
All normals point outwards:      True
No duplicate faces               True
No duplicate points              True

----- BONE-CARTILAGE -----
All interface nodes present and identical     True


In [7]:
combined_mesh_flipped.plot(scalars='region_id')

Widget(value='<iframe src="http://localhost:52577/index.html?ui=P_0x17e24ed50_1&reconnect=auto" class="pyvista…